# Prepare Calendar Table

Builds a standard calendar/date-dimension table `adb_caaresync.silver.calendar` covering **1940-01-01 to 2030-12-31**.

In [0]:
%sql
CREATE OR REPLACE TABLE adb_caaresync.silver.calendar AS

WITH date_spine AS (
  SELECT DATE_ADD('1940-01-01', CAST(id AS INT)) AS date
  FROM   RANGE(DATEDIFF('2030-12-31', '1940-01-01') + 1)
)

SELECT
  -- Key & core date
  CAST(DATE_FORMAT(date, 'yyyyMMdd') AS INT)           AS date_key,
  date,

  -- Year / Quarter / Month
  YEAR(date)                                           AS year,
  QUARTER(date)                                        AS quarter,
  CASE WHEN QUARTER(date) <= 2 THEN 1 ELSE 2 END      AS half_year,
  MONTH(date)                                          AS month,
  DATE_FORMAT(date, 'MMMM')                            AS month_name,
  DATE_FORMAT(date, 'MMM')                             AS month_name_short,
  DATE_FORMAT(date, 'yyyy-MM')                         AS year_month,

  -- Day-level
  DAY(date)                                            AS day_of_month,
  DAYOFYEAR(date)                                      AS day_of_year,
  DAYOFWEEK(date)                                      AS day_of_week,          -- 1=Sun … 7=Sat
  (DAYOFWEEK(date) + 5) % 7 + 1                        AS day_of_week_iso,      -- 1=Mon … 7=Sun
  DATE_FORMAT(date, 'EEEE')                            AS day_name,
  DATE_FORMAT(date, 'EEE')                             AS day_name_short,

  -- Week
  WEEKOFYEAR(date)                                     AS week_of_year,
  CONCAT(LPAD(YEAR(date), 4, '0'), '-W',
         LPAD(WEEKOFYEAR(date), 2, '0'))               AS iso_year_week,

  -- Fiscal year (July start → FY = year+1 when month >= 7)
  CASE WHEN MONTH(date) >= 7
       THEN YEAR(date) + 1 ELSE YEAR(date) END         AS fiscal_year,
  CASE WHEN MONTH(date) IN (7,8,9)   THEN 1
       WHEN MONTH(date) IN (10,11,12) THEN 2
       WHEN MONTH(date) IN (1,2,3)    THEN 3
       ELSE 4 END                                      AS fiscal_quarter,

  -- Flags
  CAST(DAYOFWEEK(date) IN (1, 7) AS INT)               AS is_weekend,
  CAST(DAYOFWEEK(date) NOT IN (1, 7) AS INT)           AS is_weekday,
  CAST((YEAR(date) % 4 = 0
    AND (YEAR(date) % 100 != 0 OR YEAR(date) % 400 = 0)) AS INT)
                                                       AS is_leap_year,
  CAST(date = LAST_DAY(date) AS INT)                   AS is_last_day_of_month,

  -- Month / quarter / year boundaries
  TRUNC(date, 'MONTH')                                 AS first_day_of_month,
  LAST_DAY(date)                                       AS last_day_of_month,
  TRUNC(date, 'YEAR')                                  AS first_day_of_year,
  CAST(DATE_TRUNC('QUARTER', date) AS DATE)            AS first_day_of_quarter,
  DAY(LAST_DAY(date))                                  AS days_in_month,

  -- Relative markers (vs. today)
  CAST(date = CURRENT_DATE() AS INT)                   AS is_current_day,
  CAST(YEAR(date) = YEAR(CURRENT_DATE())
   AND MONTH(date) = MONTH(CURRENT_DATE()) AS INT)     AS is_current_month,
  CAST(YEAR(date) = YEAR(CURRENT_DATE()) AS INT)       AS is_current_year

FROM date_spine

In [0]:
%sql
SELECT * FROM adb_caaresync.silver.calendar LIMIT 10